# Evaluation on DeepSeek-LLM-7B using BBEH benchmark for reasoning tasks

In [1]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 18.0 MB/s eta 0:00:00


In [2]:
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import classification_report

In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
model_name = "deepseek-ai/deepseek-llm-7b-chat"

In [5]:
bitsandbytes_config = BitsAndBytesConfig(load_in_4bit=True,
                                         bnb_4bit_compute_dtype=torch.float16,
                                         bnb_4bit_quant_type='nf4',
                                         bnb_4bit_use_double_quant=True)

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model= AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=bitsandbytes_config,
                                             device_map="auto",
                                             offload_folder="offload")

config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [7]:
def load_json(path):
  with open(path,"r") as f:
    data = json.load(f)

    df = pd.DataFrame(data['examples'])
    return df

In [8]:
def get_prompt(instruction,questions):
  prompts=[]
  for i in range(len(questions)):
    question = questions[i].split("Reply")[0]
    prompt = f"{instruction}\n\n{question}\n\nAnswer: "
    prompts.append(prompt)
  return prompts

In [20]:
def training(prompts, num_tokens=1):
    predictions = []
    for i, prompt in enumerate(prompts):
        chat = [
            {"role": "user", "content": prompt}
        ]


        formatted_prompt = tokenizer.apply_chat_template(
            chat,
            tokenize=False,
            add_generation_prompt=True
        )


        inputs = tokenizer(
            formatted_prompt,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to("cuda")

        with torch.no_grad():
            output_tokens = model.generate(
                **inputs,
                max_new_tokens=num_tokens,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.1,
                num_beams=1,
            )

        input_length = inputs["input_ids"].shape[1]
        new_tokens = output_tokens[0][input_length:]

        response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        print(f"Iteration {i}")
        print(f"Model Output: {response}\n")

        predictions.append(response)

    return predictions

## Task Casual Understanding

In [11]:
dataset = load_json("/content/task_casual_understanding.json")

In [12]:
dataset

,input,target
0,Question: Lauren and Jane work for the same co...,Yes
1,Question: Lauren and Jane work for the same co...,Yes
2,Question: The receptionist in the philosophy d...,Yes
3,Question: The receptionist in the philosophy d...,No
4,Question: Claire's parents bought her an old c...,Yes
...,...,...
195,Question: A test tube contains a mixture. Sam ...,Yes
196,Question: Reagent X and Y are both added to a ...,No
197,Question: There is a test tube on the left and...,Yes
198,Question: Sam is heating Reagent X in a test t...,Yes


In [13]:
questions = dataset['input'].values.tolist()

In [14]:
true_answers = dataset['target'].values.tolist()

In [15]:
instruction = f"""Instruction: You are an AI assistant for casual understanding. Read the story and answer with exactly one word:
Yes (if you think majority of people will answer yes), No (if you think majority of people will answer no), or Ambiguous(If you think that majority of people do not know the answer) . Do not add explanations.
"""

In [16]:
prompts = get_prompt(instruction,questions)

In [17]:
prompts[0]

"Instruction: You are an AI assistant for casual understanding. Read the story and answer with exactly one word:\nYes (if you think majority of people will answer yes), No (if you think majority of people will answer no), or Ambiguous(If you think that majority of people do not know the answer) . Do not add explanations.\n\n\nQuestion: Lauren and Jane work for the same company. They each need to use a computer for work sometimes. Unfortunately, the computer isn't very powerful. If two people are logged on at the same time, it usually crashes. So the company decided to institute an official policy. It declared that Lauren would be the only one permitted to use the computer in the mornings and that Jane would be the only one permitted to use the computer in the afternoons. As expected, Lauren logged on the computer the next day at 9:00 am. But Jane decided to disobey the official policy. She also logged on at 9:00 am. The computer crashed immediately. Did Jane cause the computer to crash

In [22]:
predictions = training(prompts,num_tokens=1)

Iteration 0
Model Output: No

Iteration 1
Model Output: No

Iteration 2
Model Output: No

Iteration 3
Model Output: No

Iteration 4
Model Output: No

Iteration 5
Model Output: No

Iteration 6
Model Output: No

Iteration 7
Model Output: No

Iteration 8
Model Output: No

Iteration 9
Model Output: No

Iteration 10
Model Output: No

Iteration 11
Model Output: No

Iteration 12
Model Output: No

Iteration 13
Model Output: No

Iteration 14
Model Output: Yes

Iteration 15
Model Output: No

Iteration 16
Model Output: Yes

Iteration 17
Model Output: No

Iteration 18
Model Output: No

Iteration 19
Model Output: No

Iteration 20
Model Output: Yes

Iteration 21
Model Output: No

Iteration 22
Model Output: No

Iteration 23
Model Output: Yes

Iteration 24
Model Output: No

Iteration 25
Model Output: Yes

Iteration 26
Model Output: No

Iteration 27
Model Output: Yes

Iteration 28
Model Output: No

Iteration 29
Model Output: No

Iteration 30
Model Output: No

Iteration 31
Model Output: No

Iteration 32

In [23]:
predictions

['No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'Yes',
 'No',
 'Yes',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Amb',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Amb',
 'Amb',
 'No',
 'No',
 'Amb',
 'Amb',
 'Yes',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'Yes',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'N

In [24]:
predictions = [elem if elem=="Yes" or elem=="No" else "Ambiguous" for elem in predictions]
predictions[:10]

['No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No']

In [26]:
predictions

['No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'Yes',
 'No',
 'Yes',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Ambiguous',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Ambiguous',
 'Ambiguous',
 'No',
 'No',
 'Ambiguous',
 'Ambiguous',
 'Yes',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'Yes',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'No',
 '

In [25]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

   Ambiguous       0.80      0.09      0.16        45
          No       0.41      0.84      0.55        80
         Yes       0.52      0.23      0.31        75

    accuracy                           0.44       200
   macro avg       0.58      0.38      0.34       200
weighted avg       0.54      0.44      0.38       200



## Task Movie Recommendation

In [27]:
dataset = load_json("/content/task_movie_recommendation.json")

In [28]:
dataset

,input,target
0,Which option has more similar movies in terms ...,(E)
1,Which option has more similar movies in terms ...,(C)
2,Which option has more similar movies in terms ...,(C)
3,Which option has more similar movies in terms ...,(A)
4,Which option has more similar movies in terms ...,(I)
...,...,...
195,Which option has more similar movies in terms ...,(E)
196,Which option has more similar movies in terms ...,(E)
197,Which option has more similar movies in terms ...,(E)
198,Which option has more similar movies in terms ...,(A)


In [29]:
dataset["target"].value_counts()

,count
target,
(A),26
(G),22
(H),21
(E),21
(J),20
(D),20
(C),18
(F),18
(B),17


In [30]:
questions = dataset['input'].values.tolist()

In [31]:
true_answers = dataset['target'].values.tolist()

In [32]:
true_answers = [x.strip("()").upper() for x in true_answers]
true_answers[:10]

['E', 'C', 'C', 'A', 'I', 'D', 'B', 'F', 'D', 'H']

In [33]:
instruction = f"""You are an assistant that determines which group of movies is most similar in terms of what people are likely to enjoy together. Reply only with the letter (A, B, C, D, E, F, G, H, I, or J) of the group that is most similar.
Do not write any explanation, full movie names, or extra text. The answer must be a single uppercase letter."""

In [34]:
prompts = get_prompt(instruction,questions)

In [35]:
prompts[0]

'You are an assistant that determines which group of movies is most similar in terms of what people are likely to enjoy together. Reply only with the letter (A, B, C, D, E, F, G, H, I, or J) of the group that is most similar.\nDo not write any explanation, full movie names, or extra text. The answer must be a single uppercase letter.\n\nWhich option has more similar movies in terms of whether a group of people will like them or not?\nOptions:\n(A) The Mask, The Lion King, Pretty Woman, Batman, The Front Page\n(B) Maelstrom, The Lion King, Lamerica, Pretty Woman, The Front Page\n(C) The Fugitive, The Mask, Maelstrom, Pretty Woman, Batman\n(D) The Lion King, The Mask, Lamerica, Maelstrom, Batman\n(E) Batman, The Lion King, Pretty Woman, The Fugitive, The Mask\n(F) Batman, The Mask, Maelstrom, The Fugitive, Lamerica\n(G) The Front Page, The Fugitive, Maelstrom, Lamerica, Batman\n(H) The Fugitive, Pretty Woman, Lamerica, Batman, The Lion King\n(I) The Fugitive, Lamerica, The Mask, The Fron

In [36]:
predictions = training(prompts,num_tokens=1)

Iteration 0
Model Output: D

Iteration 1
Model Output: E

Iteration 2
Model Output: J

Iteration 3
Model Output: F

Iteration 4
Model Output: H

Iteration 5
Model Output: F

Iteration 6
Model Output: F

Iteration 7
Model Output: F

Iteration 8
Model Output: D

Iteration 9
Model Output: G

Iteration 10
Model Output: F

Iteration 11
Model Output: E

Iteration 12
Model Output: I

Iteration 13
Model Output: I

Iteration 14
Model Output: G

Iteration 15
Model Output: G

Iteration 16
Model Output: J

Iteration 17
Model Output: J

Iteration 18
Model Output: J

Iteration 19
Model Output: IJ

Iteration 20
Model Output: F

Iteration 21
Model Output: F

Iteration 22
Model Output: J

Iteration 23
Model Output: H

Iteration 24
Model Output: H

Iteration 25
Model Output: J

Iteration 26
Model Output: B

Iteration 27
Model Output: D

Iteration 28
Model Output: H

Iteration 29
Model Output: E

Iteration 30
Model Output: G

Iteration 31
Model Output: G

Iteration 32
Model Output: D

Iteration 33
Model 

In [37]:
predictions

['D',
 'E',
 'J',
 'F',
 'H',
 'F',
 'F',
 'F',
 'D',
 'G',
 'F',
 'E',
 'I',
 'I',
 'G',
 'G',
 'J',
 'J',
 'J',
 'IJ',
 'F',
 'F',
 'J',
 'H',
 'H',
 'J',
 'B',
 'D',
 'H',
 'E',
 'G',
 'G',
 'D',
 'F',
 'E',
 'J',
 'G',
 'E',
 'F',
 'J',
 'G',
 'I',
 'J',
 'C',
 'F',
 'F',
 'G',
 'F',
 'I',
 'G',
 'D',
 'G',
 'I',
 'F',
 'I',
 'H',
 'G',
 'H',
 'G',
 'F',
 'J',
 'I',
 'F',
 'F',
 'B',
 'F',
 'H',
 'F',
 'J',
 'D',
 'F',
 'H',
 'H',
 'D',
 'F',
 'F',
 'J',
 'C',
 'F',
 'E',
 'C',
 'E',
 'F',
 'H',
 'H',
 'F',
 'D',
 'F',
 'J',
 'F',
 'F',
 'J',
 'H',
 'F',
 'G',
 'I',
 'F',
 'F',
 'F',
 'F',
 'F',
 'F',
 'E',
 'I',
 'E',
 'F',
 'H',
 'E',
 'G',
 'H',
 'F',
 'I',
 'J',
 'F',
 'D',
 'H',
 'F',
 'F',
 'E',
 'J',
 'F',
 'D',
 'D',
 'J',
 'G',
 'E',
 'H',
 'I',
 'G',
 'F',
 'J',
 'E',
 'F',
 'I',
 'E',
 'F',
 'F',
 'F',
 'E',
 'F',
 'J',
 'D',
 'J',
 'D',
 'G',
 'E',
 'J',
 'G',
 'H',
 'E',
 'J',
 'G',
 'F',
 'F',
 'H',
 'G',
 'G',
 'G',
 'F',
 'F',
 'H',
 'J',
 'F',
 'J',
 'J',
 'F',
 'I

In [41]:
predictions[19] = 'I'

In [42]:
predictions

['D',
 'E',
 'J',
 'F',
 'H',
 'F',
 'F',
 'F',
 'D',
 'G',
 'F',
 'E',
 'I',
 'I',
 'G',
 'G',
 'J',
 'J',
 'J',
 'I',
 'F',
 'F',
 'J',
 'H',
 'H',
 'J',
 'B',
 'D',
 'H',
 'E',
 'G',
 'G',
 'D',
 'F',
 'E',
 'J',
 'G',
 'E',
 'F',
 'J',
 'G',
 'I',
 'J',
 'C',
 'F',
 'F',
 'G',
 'F',
 'I',
 'G',
 'D',
 'G',
 'I',
 'F',
 'I',
 'H',
 'G',
 'H',
 'G',
 'F',
 'J',
 'I',
 'F',
 'F',
 'B',
 'F',
 'H',
 'F',
 'J',
 'D',
 'F',
 'H',
 'H',
 'D',
 'F',
 'F',
 'J',
 'C',
 'F',
 'E',
 'C',
 'E',
 'F',
 'H',
 'H',
 'F',
 'D',
 'F',
 'J',
 'F',
 'F',
 'J',
 'H',
 'F',
 'G',
 'I',
 'F',
 'F',
 'F',
 'F',
 'F',
 'F',
 'E',
 'I',
 'E',
 'F',
 'H',
 'E',
 'G',
 'H',
 'F',
 'I',
 'J',
 'F',
 'D',
 'H',
 'F',
 'F',
 'E',
 'J',
 'F',
 'D',
 'D',
 'J',
 'G',
 'E',
 'H',
 'I',
 'G',
 'F',
 'J',
 'E',
 'F',
 'I',
 'E',
 'F',
 'F',
 'F',
 'E',
 'F',
 'J',
 'D',
 'J',
 'D',
 'G',
 'E',
 'J',
 'G',
 'H',
 'E',
 'J',
 'G',
 'F',
 'F',
 'H',
 'G',
 'G',
 'G',
 'F',
 'F',
 'H',
 'J',
 'F',
 'J',
 'J',
 'F',
 'I'

In [43]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

           A       0.00      0.00      0.00        26
           B       0.50      0.06      0.11        17
           C       0.67      0.11      0.19        18
           D       0.40      0.30      0.34        20
           E       0.22      0.19      0.21        21
           F       0.15      0.50      0.23        18
           G       0.23      0.32      0.26        22
           H       0.32      0.33      0.33        21
           I       0.35      0.35      0.35        17
           J       0.16      0.25      0.19        20

    accuracy                           0.23       200
   macro avg       0.30      0.24      0.22       200
weighted avg       0.28      0.23      0.22       200



## Task Disambiguation QA

In [44]:
dataset = load_json("/content/task_disambiguationQA.json")

In [45]:
dataset

,input,target
0,Here is a sentence with pronoun(s) whose antec...,(A)
1,Here is a sentence with pronoun(s) whose antec...,(D)
2,Here is a sentence with pronoun(s) whose antec...,(E)
3,Here is a sentence with pronoun(s) whose antec...,(B)
4,Here is a sentence with pronoun(s) whose antec...,(A)
...,...,...
115,Here is a sentence with pronoun(s) whose antec...,(D)
116,Here is a sentence with pronoun(s) whose antec...,(A)
117,Here is a sentence with pronoun(s) whose antec...,(B)
118,Here is a sentence with pronoun(s) whose antec...,(D)


In [46]:
dataset['target'].value_counts()

,count
target,
(B),25
(D),24
(C),23
(A),22
(E),17
(F),7
(G),1
(H),1


In [47]:
questions = dataset['input'].values.tolist()

In [48]:
questions[0]

'Here is a sentence with pronoun(s) whose antecedent(s) can either be derived from the context or is ambiguous.\nSentence: The visitor tells the teacher that she likes the cake, and then tells the chef that it was her favorite dish.\nWhich of the following options correctly explains the antecedent(s) of the pronoun(s)?\n(A) The visitor likes cake and the visitor told the chef.\n(B) The teacher likes cake and the teacher told the chef.\n(C) The visitor likes cake and the teacher told the chef.\n(D) The teacher likes cake and the visitor told the chef.\n(E) Ambiguous.'

In [49]:
true_answers = dataset['target'].values.tolist()

In [50]:
true_answers = [x.strip("()").upper() for x in true_answers]
true_answers[:10]

['A', 'D', 'E', 'B', 'A', 'B', 'B', 'A', 'C', 'D']

In [51]:
instruction = f"""Instruction: You are a linguistics expert specializing in coreference resolution.
Task: Read the sentence and the provided options. Identify the correct antecedent(s) for the pronouns based strictly on the context and causal logic provided.
Constraint: Output ONLY the uppercase letter corresponding to the correct option. Do not explain your reasoning.
"""

In [52]:
prompts = get_prompt(instruction,questions)

In [53]:
prompts[1]

'Instruction: You are a linguistics expert specializing in coreference resolution. \nTask: Read the sentence and the provided options. Identify the correct antecedent(s) for the pronouns based strictly on the context and causal logic provided.\nConstraint: Output ONLY the uppercase letter corresponding to the correct option. Do not explain your reasoning.\n\n\nHere is a sentence with pronoun(s) whose antecedent(s) can either be derived from the context or is ambiguous.\nSentence: The designer called the janitor and asked him to clean the room as soon as possible, but he went to fetch a cup of coffee first and then returned to his desk.\nWhich of the following options correctly explains the antecedent(s) of the pronoun(s)?\n(A) Designer is cleaning the room as soon as possible and got coffee.\n(B) Janitor is cleaning the room as soon as possible and got coffee.\n(C) Designer is cleaning the room as soon as possible and janitor got coffee.\n(D) Janitor is cleaning the room as soon as pos

In [54]:
predictions = training(prompts,num_tokens=1)

Iteration 0
Model Output: C

Iteration 1
Model Output: D

Iteration 2
Model Output: D

Iteration 3
Model Output: D

Iteration 4
Model Output: B

Iteration 5
Model Output: B

Iteration 6
Model Output: B

Iteration 7
Model Output: A

Iteration 8
Model Output: B

Iteration 9
Model Output: B

Iteration 10
Model Output: C

Iteration 11
Model Output: D

Iteration 12
Model Output: B

Iteration 13
Model Output: B

Iteration 14
Model Output: A

Iteration 15
Model Output: D

Iteration 16
Model Output: D

Iteration 17
Model Output: C

Iteration 18
Model Output: D

Iteration 19
Model Output: D

Iteration 20
Model Output: The

Iteration 21
Model Output: D

Iteration 22
Model Output: D

Iteration 23
Model Output: C

Iteration 24
Model Output: B

Iteration 25
Model Output: None

Iteration 26
Model Output: A

Iteration 27
Model Output: D

Iteration 28
Model Output: D

Iteration 29
Model Output: None

Iteration 30
Model Output: None

Iteration 31
Model Output: DH

Iteration 32
Model Output: The

Iterat

In [55]:
predictions

['C',
 'D',
 'D',
 'D',
 'B',
 'B',
 'B',
 'A',
 'B',
 'B',
 'C',
 'D',
 'B',
 'B',
 'A',
 'D',
 'D',
 'C',
 'D',
 'D',
 'The',
 'D',
 'D',
 'C',
 'B',
 'None',
 'A',
 'D',
 'D',
 'None',
 'None',
 'DH',
 'The',
 'D',
 'D',
 'B',
 'None',
 'D',
 'E',
 'C',
 'D',
 'None',
 'D',
 'F',
 'C',
 'D',
 'E',
 'None',
 'None',
 'B',
 'G',
 'None',
 'None',
 'B',
 'D',
 'F',
 'B',
 'B',
 'None',
 'None',
 'DH',
 'D',
 'C',
 'B',
 'B',
 '(',
 'B',
 'A',
 'C',
 'B',
 'C',
 'A',
 'C',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'C',
 'B',
 'None',
 'C',
 'B',
 'B',
 'A',
 'C',
 'B',
 'B',
 'None',
 'The',
 'B',
 'The',
 'The',
 'B',
 'B',
 'B',
 'C',
 'B',
 'D',
 'D',
 'B',
 'B',
 'B',
 'B',
 'B',
 'D',
 'B',
 'D',
 'E',
 'D',
 'None',
 'D',
 'D',
 'B',
 'B',
 'D',
 'B']

In [57]:
predictions = [elem if elem  in list("ABCDEFGH") else "Unknown" for elem in predictions]
predictions

['C',
 'D',
 'D',
 'D',
 'B',
 'B',
 'B',
 'A',
 'B',
 'B',
 'C',
 'D',
 'B',
 'B',
 'A',
 'D',
 'D',
 'C',
 'D',
 'D',
 'Unknown',
 'D',
 'D',
 'C',
 'B',
 'Unknown',
 'A',
 'D',
 'D',
 'Unknown',
 'Unknown',
 'Unknown',
 'Unknown',
 'D',
 'D',
 'B',
 'Unknown',
 'D',
 'E',
 'C',
 'D',
 'Unknown',
 'D',
 'F',
 'C',
 'D',
 'E',
 'Unknown',
 'Unknown',
 'B',
 'G',
 'Unknown',
 'Unknown',
 'B',
 'D',
 'F',
 'B',
 'B',
 'Unknown',
 'Unknown',
 'Unknown',
 'D',
 'C',
 'B',
 'B',
 'Unknown',
 'B',
 'A',
 'C',
 'B',
 'C',
 'A',
 'C',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'C',
 'B',
 'Unknown',
 'C',
 'B',
 'B',
 'A',
 'C',
 'B',
 'B',
 'Unknown',
 'Unknown',
 'B',
 'Unknown',
 'Unknown',
 'B',
 'B',
 'B',
 'C',
 'B',
 'D',
 'D',
 'B',
 'B',
 'B',
 'B',
 'B',
 'D',
 'B',
 'D',
 'E',
 'D',
 'Unknown',
 'D',
 'D',
 'B',
 'B',
 'D',
 'B']

In [58]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

           A       0.17      0.05      0.07        22
           B       0.25      0.44      0.32        25
           C       0.21      0.13      0.16        23
           D       0.25      0.29      0.27        24
           E       0.33      0.06      0.10        17
           F       0.00      0.00      0.00         7
           G       0.00      0.00      0.00         1
           H       0.00      0.00      0.00         1
     Unknown       0.00      0.00      0.00         0

    accuracy                           0.19       120
   macro avg       0.13      0.11      0.10       120
weighted avg       0.22      0.19      0.18       120



## Task: Boolean Expressions

In [59]:
dataset = load_json("/content/task_boolean_expressions.json")

In [60]:
dataset

,input,target
0,"From the following five expressions, only one ...",(E)
1,"From the following five expressions, only one ...",(C)
2,"From the following five expressions, only one ...",(E)
3,"From the following five expressions, only one ...",(D)
4,"From the following five expressions, only one ...",(C)
...,...,...
195,"From the following five expressions, only one ...",(A)
196,"From the following five expressions, only one ...",(B)
197,"From the following five expressions, only one ...",(B)
198,"From the following five expressions, only one ...",(C)


In [61]:
dataset["target"].value_counts()

,count
target,
(E),43
(B),41
(C),40
(A),39
(D),37


In [62]:
questions = dataset["input"].values.tolist()

In [63]:
true_answers = dataset["target"].values.tolist()

In [64]:
true_answers = [x.strip("()").upper() for x in true_answers]
true_answers[:10]

['E', 'C', 'E', 'D', 'C', 'D', 'C', 'D', 'A', 'A']

In [65]:
questions[0]

'From the following five expressions, only one evaluates to true. Which one is it?\n\n(A) not (not (not (not (False) or (-5 - (1 / -5) is greater than -7)) or (not (False) or (-5 * -10 + -5 * -6 is greater than -4 * 6))) or (not (not (False) or (False)) or (not not not (-3 - (-2 / 4) is greater than -6) and not not (The capital of India is Mumbai.)))) and (not (not (not (The capital of the Nigeria is Abuja.) or (-7 - (-4 / 7) > 4)) or (not (-6 - (-7 / -5) > -6) and (True))) and (not not ((-1 * 9 is less than or equal to -6) or (-4 * 10 + -1 * -7 <= 9 * 10)) or not not not (not not (6 * -2 is less than or equal to 4) or not not not (-5 * 9 + 6 * -3 <= -9 * 6))))\n(B) ((not (not not not (not (not (not not not (not not (not (not (max(-3, 8, 5, -2) - min(-3, 8, 5, -2) <= 4) and (False)) or (max(-6, 8, 5, -1) - min(-6, 8, 5, -1) <= 9)) or not not not (-6 - (9 / -1) is less than or equal to 6)) and not not (The capital of Afghanistan is Kabul.)) or (1 - (5 / -2) > 4)) or (True)) and not not 

In [103]:
instruction=f"""You must output ONLY a single uppercase letter: A, B, C, D, or E. Do NOT include any other text, reasoning, explanations, or punctuation. Just one letter.

Instruction: You are a high-level logic and reasoning engine. Carefully evaluate the five nested Boolean expressions (A, B, C, D, E) provided.
Output exactly one uppercase letter (A, B, C, D, or E) as the final result.
"""

In [104]:
prompts = get_prompt(instruction,questions)

In [105]:
prompts[0]

'You must output ONLY a single uppercase letter: A, B, C, D, or E. Do NOT include any other text, reasoning, explanations, or punctuation. Just one letter.\n\nInstruction: You are a high-level logic and reasoning engine. Carefully evaluate the five nested Boolean expressions (A, B, C, D, E) provided. \nOutput exactly one uppercase letter (A, B, C, D, or E) as the final result.\n\n\nFrom the following five expressions, only one evaluates to true. Which one is it?\n\n(A) not (not (not (not (False) or (-5 - (1 / -5) is greater than -7)) or (not (False) or (-5 * -10 + -5 * -6 is greater than -4 * 6))) or (not (not (False) or (False)) or (not not not (-3 - (-2 / 4) is greater than -6) and not not (The capital of India is Mumbai.)))) and (not (not (not (The capital of the Nigeria is Abuja.) or (-7 - (-4 / 7) > 4)) or (not (-6 - (-7 / -5) > -6) and (True))) and (not not ((-1 * 9 is less than or equal to -6) or (-4 * 10 + -1 * -7 <= 9 * 10)) or not not not (not not (6 * -2 is less than or equa

In [109]:
predictions = training(prompts,num_tokens=10)

Iteration 0
Model Output: B

Iteration 1
Model Output: B

Iteration 2
Model Output: B

Iteration 3
Model Output: B

Iteration 4
Model Output: B

Iteration 5
Model Output: B

Iteration 6
Model Output: B

Iteration 7
Model Output: B

Iteration 8
Model Output: B

Iteration 9
Model Output: B

Iteration 10
Model Output: B

Iteration 11
Model Output: B

Iteration 12
Model Output: B

Iteration 13
Model Output: B

Iteration 14
Model Output: B

Iteration 15
Model Output: B

Iteration 16
Model Output: B

Iteration 17
Model Output: B

Iteration 18
Model Output: B

Iteration 19
Model Output: B

Iteration 20
Model Output: B

Iteration 21
Model Output: B

Iteration 22
Model Output: B

Iteration 23
Model Output: B

Iteration 24
Model Output: B

Iteration 25
Model Output: B

Iteration 26
Model Output: B

Iteration 27
Model Output: B

Iteration 28
Model Output: B

Iteration 29
Model Output: B

Iteration 30
Model Output: B

Iteration 31
Model Output: B

Iteration 32
Model Output: B

Iteration 33
Model O

In [110]:
predictions = [elem if elem  in list("ABCDE") else "Unknown" for elem in predictions]
predictions

['B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'Unknown',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B'

In [111]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

           A       0.00      0.00      0.00        39
           B       0.20      0.98      0.33        41
           C       0.00      0.00      0.00        40
           D       0.00      0.00      0.00        37
           E       0.00      0.00      0.00        43
     Unknown       0.00      0.00      0.00         0

    accuracy                           0.20       200
   macro avg       0.03      0.16      0.06       200
weighted avg       0.04      0.20      0.07       200

